# Representative Statements Workbench

The purpose of this notebook is to test the Agore representative statement selection and compare it to the Polis output for the same clusters. 

In [1]:
from pathlib import Path
from pprint import pprint
import sys

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", None)

REPO_ROOT = next(
    parent
    for parent in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (parent / "pyproject.toml").exists() and (parent / "reddwarf").exists()
)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import reddwarf
imported_repo_root = Path(reddwarf.__file__).resolve().parents[1]
if imported_repo_root != REPO_ROOT:
    raise RuntimeError(
        f"Notebook imported reddwarf from {imported_repo_root}, expected {REPO_ROOT}. Restart the kernel and rerun from the repo clone."
    )

from reddwarf.data_loader import Loader
from reddwarf.implementations.agora import run_pipeline as run_agora_pipeline
from reddwarf.implementations.polis import run_pipeline as run_polis_pipeline
from reddwarf.utils.stats import select_representative_statements
from reddwarf.utils.statements import process_statements


c:\Users\DELL\AppData\Local\Programs\Python\Python310\lib\site-packages\pydantic\_internal\_generate_schema.py:2264: UnsupportedFieldAttributeWarning: The 'exclude' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'exclude' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(


## Settings

- `DATA_SOURCE = "fixture"` uses local test fixtures
- `DATA_SOURCE = "polis_id"` fetches a live report by Polis report id

In [2]:
DATA_SOURCE = "fixture"  # fixture | polis_id
FIXTURE_DIR = "../../tests/fixtures/below-100-ptpts"
# POLIS_ID = "r2msmbb2bm7nmjxtftayt" (uncomment when u want to use polis_id)

RANDOM_STATE = 42
FDR_RATE = 0.10
DIVISIVE_N_RESAMPLES = 399
DIVISIVE_RANDOM_STATE = 42
STRONG_EFFECT_MIN = 1.0
STRONG_SEEN_MIN = 5
STRONG_PARTICIPATION_MIN = 1.0
STRONG_P_MAX = 0.05
TOP_N = 10

In [3]:
if DATA_SOURCE == "fixture":
    loader = Loader(filepaths=[
        f"{FIXTURE_DIR}/votes.json",
        f"{FIXTURE_DIR}/comments.json",
        f"{FIXTURE_DIR}/conversation.json",
    ])
    dataset_label = FIXTURE_DIR
elif DATA_SOURCE == "polis_id":
    loader = Loader(polis_id=POLIS_ID)
    dataset_label = POLIS_ID
else:
    raise ValueError(f"Unsupported DATA_SOURCE: {DATA_SOURCE}")

_, _, mod_out_statement_ids, meta_statement_ids = process_statements(loader.comments_data)

stmt_text = {}
for comment in loader.comments_data:
    sid = comment.get("tid", comment.get("statement_id"))
    stmt_text[int(sid)] = comment.get("txt", "")

print(f"Dataset: {dataset_label}")
print(f"Votes: {len(loader.votes_data)}")
print(f"Statements: {len(loader.comments_data)} total")
print(f"Moderated out / meta: {len(mod_out_statement_ids)}")


Dataset: ../../tests/fixtures/below-100-ptpts
Votes: 504
Statements: 44 total
Moderated out / meta: 2


## Run Agora Pipeline

In [4]:
agora_result = run_agora_pipeline(
    votes=loader.votes_data,
    mod_out_statement_ids=mod_out_statement_ids,
    meta_statement_ids=meta_statement_ids,
    random_state=RANDOM_STATE,
    fdr_rate=FDR_RATE,
    divisive_n_resamples=DIVISIVE_N_RESAMPLES,
    divisive_random_state=DIVISIVE_RANDOM_STATE,
    strong_effect_min=STRONG_EFFECT_MIN,
    strong_seen_min=STRONG_SEEN_MIN,
    strong_participation_min=STRONG_PARTICIPATION_MIN,
    strong_p_max=STRONG_P_MAX,
)

print(f"Agora groups: {sorted(agora_result.ranked_repness.keys())}")
print(f"Agora clustered participants: {int(agora_result.participants_df['to_cluster'].sum())}")


Agora groups: [0, 1, 2]
Agora clustered participants: 19


## Polis Representative Statements On The Same Cluster Source

This is the comparison between the two pipelines for representative-statement selection. It reuses Agora's grouped statement statistics and therefore keeps the cluster source fixed.


In [5]:
polis_repness_same_groups = select_representative_statements(
    grouped_stats_df=agora_result.group_comment_stats,
    mod_out_statement_ids=mod_out_statement_ids,
    pick_max=5,
    confidence=0.9,
)

print(f"Polis groups on same cluster source: {sorted(polis_repness_same_groups, key=int)}")


Polis groups on same cluster source: [0, 1, 2]


## View & Compare Clusters

In [6]:
def truncate(text, max_len=100):
    text = text or ""
    return text if len(text) <= max_len else text[:max_len] + "..."

def vote_breakdown(na, nd, ns):
    na = int(na)
    nd = int(nd)
    ns = int(ns)
    npass = max(ns - na - nd, 0)
    votes = f"{na}/{nd}/{npass}/{ns}"
    if ns == 0:
        pct = "0.0%/0.0%/0.0%"
    else:
        pct = f"{100*na/ns:.1f}%/{100*nd/ns:.1f}%/{100*npass/ns:.1f}%"
    return votes, pct

def polis_group_df(group_id):
    rows = []
    for rank, row in enumerate(polis_repness_same_groups.get(group_id, []), start=1):
        statement_id = int(row["tid"])
        grouped_row = agora_result.group_comment_stats.loc[(group_id, statement_id)]
        in_votes, in_pct = vote_breakdown(grouped_row["na"], grouped_row["nd"], grouped_row["ns"])
        rows.append({
            "rank": rank,
            "statement_id": statement_id,
            "repful_for": row.get("repful-for"),
            "best_agree": bool(row.get("best-agree", False)),
            "n_success": int(row["n-success"]),
            "n_trials": int(row["n-trials"]),
            "p_success": float(row["p-success"]),
            "p_test": float(row["p-test"]),
            "repness": float(row["repness"]) if row.get("repness") is not None else None,
            "repness_test": float(row["repness-test"]) if row.get("repness-test") is not None else None,
            "In Votes": in_votes,
            "In %": in_pct,
            "text": truncate(stmt_text.get(statement_id, "?")),
        })
    return pd.DataFrame(rows)

def agora_group_df_detailed(group_id, top_n=None):
    rows = []
    statements = agora_result.ranked_repness[group_id]
    if top_n is not None:
        statements = statements[:top_n]
    for statement in statements:
        npass = int(statement.ns - statement.na - statement.nd)
        npass_out = int(statement.ns_out - statement.na_out - statement.nd_out)
        in_votes, in_pct = vote_breakdown(statement.na, statement.nd, statement.ns)
        out_votes, out_pct = vote_breakdown(statement.na_out, statement.nd_out, statement.ns_out)
        rows.append({
            "rank": int(statement.rank),
            "st_id": int(statement.statement_id),
            "repful_for": statement.repful_for,
            "selected": bool(statement.selected),
            "strength": statement.signal_strength,
            "effect_size": float(statement.effect_size),
            "p_value": float(statement.p_value),
            "adjusted_p_value": float(statement.adjusted_p_value),
            "agree_effect": float(statement.agree_effect),
            "disagree_effect": float(statement.disagree_effect),
            "divisive_effect": float(statement.divisive_effect),
            "na": int(statement.na),
            "nd": int(statement.nd),
            "npass": npass,
            "ns": int(statement.ns),
            "In Votes": in_votes,
            "In %": in_pct,
            "na_out": int(statement.na_out),
            "nd_out": int(statement.nd_out),
            "npass_out": npass_out,
            "ns_out": int(statement.ns_out),
            "Out Votes": out_votes,
            "Out %": out_pct,
            "divisiveness": float(statement.divisiveness),
            "text": truncate(stmt_text.get(statement.statement_id, "?")),
        })
    return pd.DataFrame(rows)

print(f"Dataset: {dataset_label}")
for gid in sorted(agora_result.ranked_repness):
    print(f"\n=== Group {gid} ===\n")
    print("Polis representative statements (same cluster source):")
    display(polis_group_df(gid))
    print("\nAgora representative statements:")
    display(agora_group_df_detailed(gid, top_n=TOP_N))


Dataset: ../../tests/fixtures/below-100-ptpts

=== Group 0 ===

Polis representative statements (same cluster source):


,rank,statement_id,repful_for,best_agree,n_success,n_trials,p_success,p_test,repness,repness_test,In Votes,In %,text
0,1,1,agree,True,3,3,0.800000,2.000000,5.600000,3.096731,3/0/0/3,100.0%/0.0%/0.0%,The prominent display of divisive issues is concerning as politicians like to find wedge issues and ...
1,2,13,disagree,False,3,3,0.800000,2.000000,6.000000,3.207135,0/3/0/3,0.0%/100.0%/0.0%,Polis statements can be too simplistic and lack context for me to definitively vote one way or anoth...
2,3,9,disagree,False,3,3,0.800000,2.000000,4.266667,2.947154,0/3/0/3,0.0%/100.0%/0.0%,"Due to the 140 character limit, users tend to give up on communicating complex ideas."
3,4,20,disagree,False,3,3,0.800000,2.000000,4.000000,2.842821,0/3/0/3,0.0%/100.0%/0.0%,It's hard to communicate technical and quantitative points-of-view in Polis
4,5,29,disagree,False,1,1,0.666667,1.414214,5.333333,2.267787,0/1/0/1,0.0%/100.0%/0.0%,whether I engage through a text bar would really depend on the topic.



Agora representative statements:


,rank,st_id,repful_for,selected,strength,effect_size,p_value,adjusted_p_value,agree_effect,disagree_effect,divisive_effect,na,nd,npass,ns,In Votes,In %,na_out,nd_out,npass_out,ns_out,Out Votes,Out %,divisiveness,text
0,1,1,agree,True,normal,16.000000,0.001957,0.035220,16.000000,0.000000,0.000000,3,0,0,3,3/0/0/3,100.0%/0.0%/0.0%,1,5,6,12,1/5/6/12,8.3%/41.7%/50.0%,0.000000,The prominent display of divisive issues is concerning as politicians like to find wedge issues and ...
1,2,13,disagree,True,normal,16.000000,0.001341,0.035220,0.000000,16.000000,0.000000,0,3,0,3,0/3/0/3,0.0%/100.0%/0.0%,9,1,3,13,9/1/3/13,69.2%/7.7%/23.1%,0.000000,Polis statements can be too simplistic and lack context for me to definitively vote one way or anoth...
2,3,9,disagree,True,normal,8.000000,0.003207,0.038486,0.000000,8.000000,0.000000,0,3,0,3,0/3/0/3,0.0%/100.0%/0.0%,7,2,5,14,7/2/5/14,50.0%/14.3%/35.7%,0.000000,"Due to the 140 character limit, users tend to give up on communicating complex ideas."
3,4,20,disagree,True,normal,8.000000,0.004472,0.040245,0.000000,8.000000,0.000000,0,3,0,3,0/3/0/3,0.0%/100.0%/0.0%,8,2,3,13,8/2/3/13,61.5%/15.4%/23.1%,0.000000,It's hard to communicate technical and quantitative points-of-view in Polis
4,5,8,agree,False,normal,4.000000,0.022750,0.105040,4.000000,0.000000,0.000000,3,0,0,3,3/0/0/3,100.0%/0.0%/0.0%,4,1,7,12,4/1/7/12,33.3%/8.3%/58.3%,0.000000,Polis needs a way to showcase it's data without having others able to edit the contents.
5,6,5,agree,False,normal,4.000000,0.022750,0.105040,4.000000,0.000000,0.000000,3,0,0,3,3/0/0/3,100.0%/0.0%/0.0%,4,0,8,12,4/0/8/12,33.3%/0.0%/66.7%,0.000000,Tooling for multiple moderators is non-existent.
6,7,22,agree,False,normal,3.555556,0.045021,0.117001,3.555556,0.222222,1.777778,2,1,0,3,2/1/0/3,66.7%/33.3%/0.0%,2,8,3,13,2/8/3/13,15.4%/61.5%/23.1%,0.666667,Polis relies on the participant to put themselves in the other person's shoes before voting
7,8,31,agree,False,normal,3.555556,0.083265,0.130327,3.555556,0.000000,0.000000,2,0,0,2,2/0/0/2,100.0%/0.0%/0.0%,2,0,2,4,2/0/2/4,50.0%/0.0%/50.0%,0.000000,Nuance in Polis is created by entering multiple statements.
8,9,24,agree,False,normal,2.666667,0.045500,0.117001,2.666667,0.000000,0.000000,3,0,0,3,3/0/0/3,100.0%/0.0%/0.0%,6,0,6,12,6/0/6/12,50.0%/0.0%/50.0%,0.000000,Polis needs to support translations for statements
9,10,30,agree,False,normal,2.370370,0.083265,0.130327,2.370370,0.000000,0.000000,2,0,0,2,2/0/0/2,100.0%/0.0%/0.0%,3,0,1,4,3/0/1/4,75.0%/0.0%/25.0%,0.000000,Polis statements are atomic/binary



=== Group 1 ===

Polis representative statements (same cluster source):


...

,rank,st_id,repful_for,selected,strength,effect_size,p_value,adjusted_p_value,agree_effect,disagree_effect,divisive_effect,na,nd,npass,ns,In Votes,In %,na_out,nd_out,npass_out,ns_out,Out Votes,Out %,divisiveness,text
0,1,6,disagree,False,normal,2.240000,0.035496,0.343128,0.070000,2.240000,1.12,1,2,2,5,1/2/2/5,20.0%/40.0%/40.0%,8,0,4,12,8/0/4/12,66.7%/0.0%/33.3%,0.4,"It's unclear if Polis' algorithms are resilient to language nuances (sarcasm, dialects, abbreviation..."
1,2,14,disagree,False,normal,2.240000,0.016806,0.343128,0.000000,2.240000,0.00,0,2,2,4,0/2/2/4,0.0%/50.0%/50.0%,9,0,3,12,9/0/3/12,75.0%/0.0%/25.0%,0.0,"Polis should have a ""remind me later"" feature so busy people can go through all statements as per th..."
2,3,21,disagree,False,normal,2.240000,0.071031,0.402445,0.000000,2.240000,0.00,0,2,1,3,0/2/1/3,0.0%/66.7%/33.3%,6,1,1,8,6/1/1/8,75.0%/12.5%/12.5%,0.0,"I want to be able to share more complex, long-form ideas instead of 140 character statements"
3,4,32,agree,False,normal,1.120000,0.083265,0.402445,1.120000,0.000000,0.00,2,0,0,2,2/0/0/2,100.0%/0.0%/0.0%,2,0,1,3,2/0/1/3,66.7%/0.0%/33.3%,0.0,There should be a QR code for sharing poll
4,5,26,agree,False,normal,0.995556,0.025347,0.343128,0.995556,0.000000,0.00,4,0,0,4,4/0/0/4,100.0%/0.0%/0.0%,9,0,1,10,9/0/1/10,90.0%/0.0%/10.0%,0.0,I like how Polis relies on user interaction to analyze comments rather than machine learning.
5,6,1,disagree,False,normal,0.746667,0.308709,0.780262,0.000000,0.746667,0.00,0,2,2,4,0/2/2/4,0.0%/50.0%/50.0%,4,3,4,11,4/3/4/11,36.4%/27.3%/36.4%,0.0,The prominent display of divisive issues is concerning as politicians like to find wedge issues and ...
6,7,12,agree,False,normal,0.720000,0.179712,0.521166,0.720000,0.000000,0.00,3,0,1,4,3/0/1/4,75.0%/0.0%/25.0%,7,1,2,10,7/1/2/10,70.0%/10.0%/20.0%,0.0,Polis is a very simple tool to use for participants.
7,8,7,disagree,False,normal,0.560000,0.350201,0.780262,0.000000,0.560000,0.00,0,1,3,4,0/1/3/4,0.0%/25.0%/75.0%,4,1,5,10,4/1/5/10,40.0%/10.0%/50.0%,0.0,The Polis website is not compliant in terms of accessibility.
8,9,17,disagree,False,normal,0.560000,0.422678,0.780262,0.093333,0.560000,1.12,1,1,3,5,1/1/3/5,20.0%/20.0%/60.0%,6,1,4,11,6/1/4/11,54.5%/9.1%/36.4%,0.4,I have no way to communicate that 1 specific statement is more important to me than others
9,10,3,disagree,False,normal,0.560000,0.142041,0.506853,0.000000,0.560000,0.00,0,1,3,4,0/1/3/4,0.0%/25.0%/75.0%,6,0,4,10,6/0/4/10,60.0%/0.0%/40.0%,0.0,"Polis user-facing documentation is too verbose. Show, don't tell."



=== Group 2 ===

Polis representative statements (same cluster source):


,rank,statement_id,repful_for,best_agree,n_success,n_trials,p_success,p_test,repness,repness_test,In Votes,In %,text
0,1,13,agree,True,9,9,0.909091,3.162278,8.181818,3.783937,9/0/0/9,100.0%/0.0%/0.0%,Polis statements can be too simplistic and lack context for me to definitively vote one way or anoth...
1,2,27,agree,False,5,6,0.750000,1.889822,5.250000,2.489547,5/0/1/6,83.3%/0.0%/16.7%,I wish statements weren't limited to only 140 characters and there was more clarity about how statem...
2,3,19,agree,False,9,11,0.769231,2.309401,3.461538,2.608746,9/2/0/11,81.8%/18.2%/0.0%,Polis feels too abstract and alienating since there is no context or background to justify a positio...
3,4,9,agree,False,7,10,0.666667,1.507557,6.000000,2.595913,7/1/2/10,70.0%/10.0%/20.0%,"Due to the 140 character limit, users tend to give up on communicating complex ideas."
4,5,0,disagree,False,8,8,0.900000,3.000000,1.800000,2.178819,0/8/0/8,0.0%/100.0%/0.0%,Polis is perfect just the way it is. No changes are necessary.



Agora representative statements:


,rank,st_id,repful_for,selected,strength,effect_size,p_value,adjusted_p_value,agree_effect,disagree_effect,divisive_effect,na,nd,npass,ns,In Votes,In %,na_out,nd_out,npass_out,ns_out,Out Votes,Out %,divisiveness,text
0,1,13,agree,True,normal,5.355372,0.000154,0.005557,5.355372,0.000000,0.000000,9,0,0,9,9/0/0/9,100.0%/0.0%/0.0%,0,4,3,7,0/4/3/7,0.0%/57.1%/42.9%,0.000000,Polis statements can be too simplistic and lack context for me to definitively vote one way or anoth...
1,2,19,agree,True,strong,5.355372,0.009087,0.067925,5.355372,0.088154,0.528926,9,2,0,11,9/2/0/11,81.8%/18.2%/0.0%,1,3,3,7,1/3/3/7,14.3%/42.9%/42.9%,0.363636,Polis feels too abstract and alienating since there is no context or background to justify a positio...
2,3,20,agree,True,normal,3.239669,0.019624,0.088310,3.239669,0.088154,0.528926,7,2,0,9,7/2/0/9,77.8%/22.2%/0.0%,1,3,3,7,1/3/3/7,14.3%/42.9%/42.9%,0.363636,It's hard to communicate technical and quantitative points-of-view in Polis
3,4,9,agree,True,normal,3.239669,0.009434,0.067925,3.239669,0.016529,0.132231,7,1,2,10,7/1/2/10,70.0%/10.0%/20.0%,0,4,3,7,0/4/3/7,0.0%/57.1%/42.9%,0.181818,"Due to the 140 character limit, users tend to give up on communicating complex ideas."
4,5,17,agree,True,normal,2.380165,0.018422,0.088310,2.380165,0.000000,0.000000,6,0,2,8,6/0/2/8,75.0%/0.0%/25.0%,1,2,5,8,1/2/5/8,12.5%/25.0%/62.5%,0.000000,I have no way to communicate that 1 specific statement is more important to me than others
5,6,21,agree,False,normal,1.652893,0.029391,0.108449,1.652893,0.000000,0.000000,5,0,1,6,5/0/1/6,83.3%/0.0%/16.7%,1,3,1,5,1/3/1/5,20.0%/60.0%/20.0%,0.000000,"I want to be able to share more complex, long-form ideas instead of 140 character statements"
6,7,3,agree,False,normal,1.652893,0.130570,0.223834,1.652893,0.000000,0.000000,5,0,3,8,5/0/3/8,62.5%/0.0%/37.5%,1,1,4,6,1/1/4/6,16.7%/16.7%/66.7%,0.000000,"Polis user-facing documentation is too verbose. Show, don't tell."
7,8,2,agree,False,normal,1.652893,0.036150,0.108449,1.652893,0.132231,0.528926,5,2,1,8,5/2/1/8,62.5%/25.0%/12.5%,0,2,4,6,0/2/4/6,0.0%/33.3%/66.7%,0.363636,"There's no way to differentiate between ""disagree"" and ""pass"" votes. This is misleading as the 2 pos..."
8,9,25,agree,False,normal,1.652893,0.085804,0.191161,1.652893,0.066116,0.528926,5,2,1,8,5/2/1/8,62.5%/25.0%/12.5%,1,4,2,7,1/4/2/7,14.3%/57.1%/28.6%,0.363636,I want to be able to make a statement referring to an existing statement so I can provide more conte...
9,10,27,agree,True,normal,1.652893,0.012791,0.076744,1.652893,0.000000,0.000000,5,0,1,6,5/0/1/6,83.3%/0.0%/16.7%,0,1,4,5,0/1/4/5,0.0%/20.0%/80.0%,0.000000,I wish statements weren't limited to only 140 characters and there was more clarity about how statem...


## Summary

Comparison of Polis top 5 ids, Agora top-ranked ids and Agora `selected=True` ids per group.


In [7]:
summary_rows = []
for gid in sorted(agora_result.ranked_repness):
    polis_top_ids = [int(row["tid"]) for row in polis_repness_same_groups.get(gid, [])]
    agora_top_ids = [statement.statement_id for statement in agora_result.ranked_repness[gid][:TOP_N]]
    agora_selected_ids = [statement.statement_id for statement in agora_result.ranked_repness[gid] if statement.selected]
    summary_rows.append({
        "group_id": gid,
        "polis_top_ids": polis_top_ids,
        "agora_top_ids": agora_top_ids,
        "agora_selected_ids": agora_selected_ids,
        "n_agora_selected": len(agora_selected_ids),
    })

display(pd.DataFrame(summary_rows))


,group_id,polis_top_ids,agora_top_ids,agora_selected_ids,n_agora_selected
0,0,"[1, 13, 9, 20, 29]","[1, 13, 9, 20, 8, 5, 22, 31, 24, 30]","[1, 13, 9, 20]",4
1,1,[26],"[6, 14, 21, 32, 26, 1, 12, 7, 17, 3]",[],0
2,2,"[13, 27, 19, 9, 0]","[13, 19, 20, 9, 17, 21, 3, 2, 25, 27]","[13, 19, 20, 9, 17, 27, 0, 28]",8
